In [10]:
from langchain.agents import create_agent
from langchain_ollama import ChatOllama
from langchain.messages import HumanMessage
from ipywidgets import FileUpload
from IPython.display import display
import base64
import sounddevice as sd
from scipy.io.wavfile import write
import io
import time
from tqdm import tqdm


In [3]:
uploader = FileUpload(accept='.jpg', multiple=False)
display(uploader)

FileUpload(value=(), accept='.jpg', description='Upload')

In [4]:
print(uploader.value)

({'name': '215167.jpg', 'type': 'image/jpeg', 'size': 148720, 'content': <memory at 0x000001F269C65C00>, 'last_modified': datetime.datetime(2026, 5, 12, 3, 44, 11, 734000, tzinfo=datetime.timezone.utc)},)


In [6]:
uploaded_file = uploader.value[0]

content_mv = uploaded_file['content']

img_bytes = bytes(content_mv)

img_b64 = base64.b64encode(img_bytes).decode('utf-8')

In [8]:
multimodel_questions = HumanMessage(content=[
    {'type': 'text', 'text': 'What is this image about?'},
    {'type': 'image', 'base64': img_b64, 'mime_type': 'image/jpg'}
])
model = ChatOllama(
    model='gemma4:e4b',
    temperature=0
)
agent = create_agent(
    model=model,
)

In [9]:
response = agent.invoke({'messages':[multimodel_questions]})
print(response['messages'][-1].content)


This image is a highly stylized, dramatic illustration of a powerful **fictional warrior character**, most likely originating from the **anime/manga genre**, specifically resembling characters from the *Dragon Ball* universe or similar martial arts action series.

Here is a detailed breakdown of what the image is about:

### 🥋 Subject and Character
*   **The Character:** The central figure is a muscular male warrior with intensely spiky hair and a determined, powerful expression.
*   **Attire:** He is dressed in flowing, layered robes (primarily red and orange) secured with a dark sash. This attire suggests a martial arts background or a warrior caste.
*   **Mood:** The character exudes immense power, intensity, and readiness for combat.

### 🎨 Artistic Style and Composition
*   **Style:** The artwork is rendered in a dynamic, comic book/manga style, characterized by strong lines, dramatic shading, and high energy.
*   **Color Palette:** The palette is dominated by vibrant reds, orange

In [12]:
# Recording settings
duration = 5  # seconds
sample_rate = 44100

print("Recording...")
audio = sd.rec(int(duration * sample_rate), samplerate=sample_rate, channels=1)
# Progress bar for the duration
for _ in tqdm(range(duration * 10)):   # update 10× per second
    time.sleep(0.1)
sd.wait()
print("Done.")

# Write WAV to an in-memory buffer
buf = io.BytesIO()
write(buf, sample_rate, audio)
wav_bytes = buf.getvalue()

aud_b64 = base64.b64encode(wav_bytes).decode("utf-8")

Recording...


100%|██████████| 50/50 [00:05<00:00,  9.78it/s]

Done.


In [13]:
multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "Tell me about this audio file"},
    {"type": "audio", "base64": aud_b64, "mime_type": "audio/wav"}
])

response = agent.invoke(
    {"messages": [multimodal_question]}
)

print(response['messages'][-1].content)

ValueError: Blocks of type audio not supported.